# 🏥 Skin Disease Classification Training Notebook (Low-RAM Version)
This notebook trains a deep learning model (EfficientNetB2) on the **HAM10000** skin lesion dataset.  
It uses memory-efficient `tf.data` pipelines to stream files directly from disk, preventing memory crashes on standard Colab runtimes.

### ⚠️ Make sure GPU acceleration is active:
1. In the top menu, go to **Runtime** ➔ **Change runtime type**.
2. Select **T4 GPU** (or any active GPU) as the Hardware Accelerator.
3. Click **Save**.

## 🛠️ Step 1: Install Dependencies

In [ ]:
!pip install kagglehub tensorflow pandas scikit-learn opencv-python pillow matplotlib seaborn tqdm

## 📂 Step 2: Download and Organize Dataset

In [ ]:
import os
import shutil
import kagglehub
from tqdm import tqdm

# 1. Download dataset
print("Downloading dataset from Kaggle via kagglehub...")
path = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")
print("Raw dataset path:", path)

# 2. Create target directories
os.makedirs('./HAM10000_images', exist_ok=True)

# 3. Copy metadata CSV
shutil.copy2(os.path.join(path, "HAM10000_metadata.csv"), "./HAM10000_metadata.csv")
print("Metadata file copied successfully.")

# 4. Copy images from both parts
parts = ["HAM10000_images_part_1", "HAM10000_images_part_2"]
for part in parts:
    src_part_dir = os.path.join(path, part)
    if os.path.exists(src_part_dir):
        print(f"Copying images from {part}...")
        files = os.listdir(src_part_dir)
        for filename in tqdm(files):
            shutil.copy2(os.path.join(src_part_dir, filename), os.path.join("./HAM10000_images", filename))

print("Verification - Total images copied:", len(os.listdir("./HAM10000_images")))

## 🏗️ Step 3: Define Model and Pipeline Classes

In [ ]:
import os
import sys
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
from tqdm import tqdm

warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, callbacks
from tensorflow.keras.applications import EfficientNetB2
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
import cv2

print(f"TensorFlow {tf.__version__} loaded successfully")
print(f"GPUs Available: {len(tf.config.list_physical_devices('GPU'))}")

class Config:
    DATASET_PATH = './HAM10000_images'
    METADATA_PATH = './HAM10000_metadata.csv'
    MODEL_SAVE_PATH = './skin_disease_model.h5'
    
    IMAGE_SIZE = 224
    BATCH_SIZE = 32
    EPOCHS = 50
    LEARNING_RATE = 1e-4
    DROPOUT_RATE = 0.3
    
    VALIDATION_SPLIT = 0.15
    TEST_SPLIT = 0.15
    PATIENCE = 10
    MIN_LR = 1e-7
    LR_FACTOR = 0.5
    LR_PATIENCE = 5
    
    DISEASE_CLASSES = {
        'akiec': 'Actinic Keratoses',
        'bcc':   'Basal Cell Carcinoma',
        'bkl':   'Benign Keratosis',
        'df':    'Dermatofibroma',
        'mel':   'Melanoma',
        'nv':    'Melanocytic Nevi',
        'vasc':  'Vascular Lesions',
    }
    CLASS_NAMES = sorted(DISEASE_CLASSES.keys())
    NUM_CLASSES = len(CLASS_NAMES)
    
    COLORS = {
        'mel':   '#FF4444',
        'bcc':   '#FF8800',
        'akiec': '#FFAA00',
        'bkl':   '#FFDD00',
        'nv':    '#44BB44',
        'df':    '#44BBBB',
        'vasc':  '#4488FF',
    }
    RANDOM_SEED = 42

class SkinDataLoader:
    def __init__(self, config=None):
        self.config = config or Config()
        self.metadata = None
        self.class_weights = None
        
    def load_metadata(self):
        path = self.config.METADATA_PATH
        self.metadata = pd.read_csv(path)
        print(f"Metadata loaded: {len(self.metadata)} entries")
        self.metadata['filepath'] = self.metadata['image_id'].apply(self.get_image_path)
        self.metadata = self.metadata.dropna(subset=['filepath']).reset_index(drop=True)
        self.metadata['label_idx'] = self.metadata['dx'].apply(lambda x: self.config.CLASS_NAMES.index(x))
        
        y_labels = self.metadata['label_idx'].values
        class_w = compute_class_weight('balanced', classes=np.unique(y_labels), y=y_labels)
        self.class_weights = dict(enumerate(class_w))
        return self.metadata
    
    def get_image_path(self, image_id):
        for ext in ['.jpg', '.jpeg', '.png']:
            path = os.path.join(self.config.DATASET_PATH, image_id + ext)
            if os.path.exists(path):
                return path
        return None
    
    def split_and_build_datasets(self):
        if self.metadata is None:
            self.load_metadata()
        train_df, temp_df = train_test_split(
            self.metadata,
            test_size=self.config.VALIDATION_SPLIT + self.config.TEST_SPLIT,
            random_state=self.config.RANDOM_SEED,
            stratify=self.metadata['label_idx']
        )
        rel_test = self.config.TEST_SPLIT / (self.config.VALIDATION_SPLIT + self.config.TEST_SPLIT)
        val_df, test_df = train_test_split(
            temp_df,
            test_size=rel_test,
            random_state=self.config.RANDOM_SEED,
            stratify=temp_df['label_idx']
        )
        
        # Define a local map function to avoid serializing self (pickling module error)
        image_size = self.config.IMAGE_SIZE
        def parse_image_fn(filepath, label):
            img = tf.io.read_file(filepath)
            img = tf.image.decode_jpeg(img, channels=3)
            img = tf.image.resize(img, [image_size, image_size])
            img = tf.cast(img, tf.float32) / 255.0
            return img, label

        train_ds = tf.data.Dataset.from_tensor_slices((train_df['filepath'].values, train_df['label_idx'].values))
        train_ds = train_ds.shuffle(buffer_size=2048, seed=self.config.RANDOM_SEED)
        train_ds = train_ds.map(parse_image_fn, num_parallel_calls=tf.data.AUTOTUNE)
        train_ds = train_ds.batch(self.config.BATCH_SIZE)
        train_ds = train_ds.prefetch(buffer_size=tf.data.AUTOTUNE)
        
        val_ds = tf.data.Dataset.from_tensor_slices((val_df['filepath'].values, val_df['label_idx'].values))
        val_ds = val_ds.map(parse_image_fn, num_parallel_calls=tf.data.AUTOTUNE)
        val_ds = val_ds.batch(self.config.BATCH_SIZE)
        val_ds = val_ds.prefetch(buffer_size=tf.data.AUTOTUNE)
        
        test_ds = tf.data.Dataset.from_tensor_slices((test_df['filepath'].values, test_df['label_idx'].values))
        test_ds = test_ds.map(parse_image_fn, num_parallel_calls=tf.data.AUTOTUNE)
        test_ds = test_ds.batch(self.config.BATCH_SIZE)
        test_ds = test_ds.prefetch(buffer_size=tf.data.AUTOTUNE)
        
        return train_ds, val_ds, test_ds, test_df

class SkinDiseaseModel:
    def __init__(self, config=None):
        self.config = config or Config()
        self.model = None
        self.history = None
        
    def build_model_v1_efficient(self):
        base = EfficientNetB2(
            weights='imagenet',
            include_top=False,
            input_shape=(self.config.IMAGE_SIZE, self.config.IMAGE_SIZE, 3)
        )
        base.trainable = False
        inputs = keras.Input(shape=(self.config.IMAGE_SIZE, self.config.IMAGE_SIZE, 3))
        x = layers.RandomFlip("horizontal_and_vertical")(inputs)
        x = layers.RandomRotation(0.2)(x)
        x = layers.RandomZoom(0.1)(x)
        x = layers.RandomBrightness(0.2)(x)
        x = layers.RandomContrast(0.1)(x)
        x = base(x, training=False)
        x = layers.GlobalAveragePooling2D()(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dense(512, activation='relu')(x)
        x = layers.Dropout(self.config.DROPOUT_RATE)(x)
        x = layers.Dense(256, activation='relu')(x)
        x = layers.Dropout(self.config.DROPOUT_RATE)(x)
        outputs = layers.Dense(self.config.NUM_CLASSES, activation='softmax')(x)
        self.model = Model(inputs, outputs, name='SkinDiseaseClassifier_EfficientNet')
        self._compile_model()
        return self.model
        
    def _compile_model(self):
        self.model.compile(
            optimizer=Adam(learning_rate=self.config.LEARNING_RATE),
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy',
                     tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3, name='top3_accuracy')]
        )
        
    def _get_callbacks(self):
        return [
            callbacks.EarlyStopping(monitor='val_loss', patience=self.config.PATIENCE, restore_best_weights=True),
            callbacks.ReduceLROnPlateau(monitor='val_loss', factor=self.config.LR_FACTOR, patience=self.config.LR_PATIENCE, min_lr=self.config.MIN_LR),
            callbacks.ModelCheckpoint(filepath=self.config.MODEL_SAVE_PATH.replace('.h5', '_best.h5'), monitor='val_accuracy', save_best_only=True)
        ]
        
    def train(self, train_ds, val_ds, class_weights=None):
        self.history = self.model.fit(
            train_ds,
            epochs=self.config.EPOCHS,
            validation_data=val_ds,
            class_weight=class_weights,
            callbacks=self._get_callbacks()
        )
        
        # Fine-tuning
        for layer in self.model.layers:
            if hasattr(layer, 'trainable'):
                layer.trainable = True
        self.model.compile(
            optimizer=Adam(learning_rate=self.config.LEARNING_RATE * 0.1),
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy', tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3, name='top3_accuracy')]
        )
        self.model.fit(
            train_ds,
            epochs=20,
            validation_data=val_ds,
            class_weight=class_weights,
            callbacks=[callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)]
        )
        return self.history
        
    def evaluate(self, test_ds, y_test):
        loss, acc, top3 = self.model.evaluate(test_ds, verbose=0)
        y_pred_prob = self.model.predict(test_ds)
        y_pred = np.argmax(y_pred_prob, axis=1)
        return {
            'loss': float(loss),
            'accuracy': float(acc),
            'top3_accuracy': float(top3),
            'y_pred': y_pred
        }
        
    def save_model(self):
        self.model.save(self.config.MODEL_SAVE_PATH)
        cfg_path = self.config.MODEL_SAVE_PATH.replace('.h5', '_config.json')
        cfg_data = {
            'class_names': self.config.CLASS_NAMES,
            'disease_classes': self.config.DISEASE_CLASSES,
            'image_size': self.config.IMAGE_SIZE,
            'num_classes': self.config.NUM_CLASSES,
        }
        with open(cfg_path, 'w') as f:
            json.dump(cfg_data, f, indent=2)

class Visualizer:
    def __init__(self, config=None):
        self.config = config or Config()
        
    def plot_training_history(self, history, save_path='training_history.png'):
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        metrics = [
            ('accuracy', 'val_accuracy', 'Accuracy'),
            ('loss', 'val_loss', 'Loss'),
            ('top3_accuracy', 'val_top3_accuracy', 'Top-3 Accuracy'),
        ]
        for ax, (train_key, val_key, title) in zip(axes, metrics):
            if train_key in history.history:
                ax.plot(history.history[train_key], label='Train')
            if val_key in history.history:
                ax.plot(history.history[val_key], label='Validation')
            ax.set_title(title)
            ax.legend()
            ax.grid(True)
        plt.savefig(save_path)
        plt.show()
        
    def plot_confusion_matrix(self, y_true, y_pred, save_path='confusion_matrix.png'):
        class_names = [self.config.DISEASE_CLASSES[c] for c in self.config.CLASS_NAMES]
        cm = confusion_matrix(y_true, y_pred)
        cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))
        sns.heatmap(cm, annot=True, fmt='d', ax=ax1, xticklabels=class_names, yticklabels=class_names, cmap='Blues')
        sns.heatmap(cm_norm, annot=True, fmt='.2f', ax=ax2, xticklabels=class_names, yticklabels=class_names, cmap='Blues')
        plt.savefig(save_path)
        plt.show()

## 🏋️ Step 4: Run Training Pipeline

In [ ]:
config = Config()
tf.random.set_seed(config.RANDOM_SEED)
np.random.seed(config.RANDOM_SEED)

# 1. Load Data
loader = SkinDataLoader(config)
train_ds, val_ds, test_ds, test_df = loader.split_and_build_datasets()

# 2. Build Model
skin_model = SkinDiseaseModel(config)
skin_model.build_model_v1_efficient()

# 3. Train
history = skin_model.train(train_ds, val_ds, class_weights=loader.class_weights)

# 4. Plot History
viz = Visualizer(config)
viz.plot_training_history(history)

# 5. Evaluate and Plot CM
results = skin_model.evaluate(test_ds, test_df['label_idx'].values)
viz.plot_confusion_matrix(test_df['label_idx'].values, results['y_pred'])

# 6. Save
skin_model.save_model()
print("Training pipeline finished successfully!")

## 📥 Step 5: Download Trained Model back to Asus ExpertBook

In [ ]:
from google.colab import files

if os.path.exists('skin_disease_model.h5'):
    print("Downloading trained Keras model...")
    files.download('skin_disease_model.h5')
    files.download('skin_disease_model_config.json')
if os.path.exists('training_history.png'):
    print("Downloading metric plots...")
    files.download('training_history.png')
    files.download('confusion_matrix.png')
print("Download popups triggered. If blocked, check your browser's address bar!")